# Лабараторная праца №7: Прагназаванне часавых радаў - Prophet vs. RNN/LSTM/GRU

Гэты ноўтбук прысвечаны параўнальнаму аналізу двух прынцыпова розных падыходаў да прагназавання часавых радаў:
1. **Класічны статыстычны / адытыўны падыход** з выкарыстаннем бібліятэкі `Prophet` ад Meta.
2. **Нейрасецевы падыход** на аснове рэкурэнтных нейронавых сетак (`LSTM`/`GRU`) з выкарыстаннем TensorFlow/Keras.

Усе тлумачэнні, каментарыі, назвы і пазнакі на графіках выкананы на **беларускай мове**.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Налада неінтэрактыўнага бэкэнда для прадухілення блакіроўкі на Windows
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Input, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from prophet import Prophet
import warnings
warnings.filterwarnings('ignore')

# Налада прайгравальнасці
np.random.seed(42)
tf.random.set_seed(42)

# Налада візуалізацыі
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
print("Бібліятэкі паспяхова імпартаваны ў неінтэрактыўным рэжыме!")

Бібліятэкі паспяхова імпартаваны ў неінтэрактыўным рэжыме!


Importing plotly failed. Interactive plots will not work.


## 1. Загрузка даных і першасны аналіз (EDA)
Мы будзем выкарыстоўваць класічны часавы рад **CO2** ад `statsmodels` (тыднёвыя вымярэнні канцэнтрацыі вуглякіслага газу ў атмасферы). Гэты рад мае ярка выражаны трэнд, гадавую сезоннасць і некалькі прапушчаных значэнняў, што робіць яго ідэальным для навучання і праверкі мадэляў.

In [2]:
# Загрузка даных
co2_data = sm.datasets.co2.load_pandas().data
print("Агульны памер загружанага рада:", co2_data.shape)
print("\nПершыя 5 радкоў:")
print(co2_data.head())
print("\nКолькасць прапушчаных значэнняў:")
print(co2_data.isna().sum())

Агульны памер загружанага рада: (2284, 1)

Першыя 5 радкоў:
              co2
1958-03-29  316.1
1958-04-05  317.3
1958-04-12  317.6
1958-04-19  317.5
1958-04-26  316.4

Колькасць прапушчаных значэнняў:
co2    59
dtype: int64


### Апрацоўка прапушчаных значэнняў
Паколькі ў радзе ёсць прапушчаныя значэнні (NaN), мы запоўнім іх з дапамогай лінейнай інтэрпаляцыі, што з'яўляецца стандартнай практыкай для часавых радаў.

In [3]:
# Лінейная інтэрпаляцыя прапушчаных значэнняў
df_cleaned = co2_data.interpolate(method='linear')
print("Колькасць прапушчаных значэнняў пасля інтэрпаляцыі:", df_cleaned.isna().sum().values[0])

# Створым асобную калонку для даты і ператворым індэкс у звычайную калонку
df_cleaned = df_cleaned.reset_index()
df_cleaned.columns = ['date', 'co2']
print("\nАбноўлены набор даных:")
print(df_cleaned.head())

Колькасць прапушчаных значэнняў пасля інтэрпаляцыі: 0

Абноўлены набор даных:
        date    co2
0 1958-03-29  316.1
1 1958-04-05  317.3
2 1958-04-12  317.6
3 1958-04-19  317.5
4 1958-04-26  316.4


### Візуалізацыя часавага рада, трэнду і сезоннасці
Зробім дэкампазіцыю часавага рада на трэнд, сезоннасць і рэшту (шум) з выкарыстаннем функцыі `seasonal_decompose` з перыядам 52 (паколькі даныя тыднёвыя, 52 тыдні = 1 год).

In [4]:
# Налада індэкса для seasonal_decompose
decomposition_data = df_cleaned.set_index('date')

# Выкананне дэкампазіцыі (перыяд 52 тыдні)
decomposition = seasonal_decompose(decomposition_data['co2'], model='additive', period=52)

# Пабудова графікаў
fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomposition.observed.plot(ax=ax1, color='royalblue')
ax1.set_ylabel('Назіраныя', fontsize=12)
ax1.set_title('Дэкампазіцыя часавага рада CO2 (тыднёвыя даныя)', fontsize=14)

decomposition.trend.plot(ax=ax2, color='darkorange')
ax2.set_ylabel('Трэнд', fontsize=12)

decomposition.seasonal.plot(ax=ax3, color='forestgreen')
ax3.set_ylabel('Сезоннасць', fontsize=12)

decomposition.resid.plot(ax=ax4, color='crimson', style='.')
ax4.set_ylabel('Залішкавы шум', fontsize=12)
ax4.set_xlabel('Дата', fontsize=12)

plt.tight_layout()
plt.savefig('seasonal_decomposition.png', dpi=300)
plt.show()

### Праверка стацыянарнасці (Тэст Дзікі-Фуллера)
Часавы рад з'яўляецца стацыянарным, калі яго статыстычныя ўласцівасці (сярэдняе, дысперсія) не змяняюцца з часам. Правядзем пашыраны тэст Дзікі-Фуллера (ADF test).
- **Нулявая гіпотэза ($H_0$):** Рад не стацыянарны (мае адзінкавы корань).
- **Альтэрнатыўная гіпотэза ($H_1$):** Рад стацыянарны.

In [5]:
# Выкананне тэсту Дзікі-Фуллера
adf_result = adfuller(df_cleaned['co2'])

print("Вынікі тэсту Дзікі-Фуллера:")
print(f"  ADF-статыстыка: {adf_result[0]:.4f}")
print(f"  p-value: {adf_result[1]:.4e}")
print("  Крытычныя значэнні:")
for key, value in adf_result[4].items():
    print(f"    {key}: {value:.4f}")

if adf_result[1] < 0.05:
    print("\nРашэнне: Нулявая гіпотэза адхіляецца. Рад з'яўляецца стацыянарным (p-value < 0.05).")
else: 
    print("\nРашэнне: Нулявая гіпотэза НЕ адхіляецца. Рад НЕ з'яўляецца стацыянарным (p-value >= 0.05). Наяўны трэнд.")

Вынікі тэсту Дзікі-Фуллера:
  ADF-статыстыка: 0.0338
  p-value: 9.6124e-01
  Крытычныя значэнні:
    1%: -3.4333
    5%: -2.8628
    10%: -2.5675

Рашэнне: Нулявая гіпотэза НЕ адхіляецца. Рад НЕ з'яўляецца стацыянарным (p-value >= 0.05). Наяўны трэнд.


## 2. Падрыхтоўка даных (Двайны падыход)

Мы падзелім нашы даныя на навучальную і тэставую выбаркі. Храналагічны падзел: першыя **80%** даных — для навучання мадэляў, а апошнія **20%** — для праверкі іх якасці (тэставання).

In [6]:
# Вызначэнне кропкі падзелу
split_idx = int(len(df_cleaned) * 0.8)
train_df = df_cleaned.iloc[:split_idx].copy()
test_df = df_cleaned.iloc[split_idx:].copy()

print(f"Памер навучальнай выбаркі: {len(train_df)} (з {train_df['date'].min().strftime('%Y-%m-%d')} па {train_df['date'].max().strftime('%Y-%m-%d')})")
print(f"Памер тэставай выбаркі: {len(test_df)} (з {test_df['date'].min().strftime('%Y-%m-%d')} па {test_df['date'].max().strftime('%Y-%m-%d')})")

Памер навучальнай выбаркі: 1827 (з 1958-03-29 па 1993-03-27)
Памер тэставай выбаркі: 457 (з 1993-04-03 па 2001-12-29)


### А. Падрыхтоўка даных для Prophet
Prophet патрабуе спецыфічны фармат: DataFrame з дзвюма калонкамі: `ds` для дат і `y` для значэнняў мэтавага паказчыка.

In [7]:
# Стварэнне датафрэймаў для Prophet
prophet_train = train_df[['date', 'co2']].rename(columns={'date': 'ds', 'co2': 'y'})
prophet_test = test_df[['date', 'co2']].rename(columns={'date': 'ds', 'co2': 'y'})

print("Прыклад даных для Prophet:")
print(prophet_train.head())

Прыклад даных для Prophet:
          ds      y
0 1958-03-29  316.1
1 1958-04-05  317.3
2 1958-04-12  317.6
3 1958-04-19  317.5
4 1958-04-26  316.4


### Б. Падрыхтоўка даных для Нейронавай сеткі (LSTM/GRU)
Рэкурэнтныя сеткі патрабуюць:
1. **Нармалізацыі даных** (напрыклад, у дыяпазон `[0, 1]`), каб пазбегнуць праблемы згасання градыентаў і паскорыць навучанне.
2. **Стварэння слізгальных вокнаў назіранняў** ($X$ — гістарычнае акно, $y$ — наступны крок). Мы возьмем акно памерам **52 тыдні** (1 год), каб сетка магла выкарыстоўваць гадавую гісторыю для прагнозу на наступны тыдзень.

In [8]:
# Нармалізацыя даных з дапамогай MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 1))

# Навучаем scaler толькі на навучальнай выбарцы, каб пазбегнуць выцеку інфармацыі (data leakage)
train_scaled = scaler.fit_transform(train_df[['co2']])
test_scaled = scaler.transform(test_df[['co2']])

# Функцыя для генерацыі слізгальных вокнаў
def create_sliding_windows(data, window_size=52):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:(i + window_size)])
        y.append(data[i + window_size])
    return np.array(X), np.array(y)

# Памер акна
WINDOW_SIZE = 52

# Стварэнне вокнаў для навучання
X_train, y_train = create_sliding_windows(train_scaled, WINDOW_SIZE)
print("Форма X_train:", X_train.shape) # (samples, window_size, features)
print("Форма y_train:", y_train.shape) # (samples, 1)

# Для тэставання нам спатрэбіцца гісторыя з канца навучальнай выбаркі, 
# каб пачаць прагназаваць першыя кропкі тэставага перыяду
# Аб'ядноўваем апошнія WINDOW_SIZE кропак навучальнай выбаркі і тэставую выбарку
full_test_data = np.vstack((train_scaled[-WINDOW_SIZE:], test_scaled)) 
X_test, y_test = create_sliding_windows(full_test_data, WINDOW_SIZE)

print("Форма X_test:", X_test.shape)
print("Форма y_test:", y_test.shape)

Форма X_train: (1775, 52, 1)
Форма y_train: (1775, 1)
Форма X_test: (457, 52, 1)
Форма y_test: (457, 1)


## 3. Рэалізацыя і навучанне Нейронавай Сеткі (GRU)
Рэкурэнтная сетка на аснове GRU (Gated Recurrent Unit) паказвае выдатныя вынікі на часавых радах і патрабуе менш вылічальных рэсурсаў, чым LSTM. Побудуем мадэль з адным пластом GRU (64 нейроны), пластом Dropout для рэгулярызацыі і Dense пластом для канчатковага прагнозу аднаго значэння.

In [9]:
# Налада прайгравальнасці
tf.random.set_seed(42)

# Побудова архітэктуры GRU мадэлі
model = Sequential([
    Input(shape=(WINDOW_SIZE, 1)),
    GRU(units=64, activation='tanh', return_sequences=False),
    Dropout(0.2),
    Dense(units=1)
])

# Кампіляцыя мадэлі
model.compile(optimizer='adam', loss='mse')
model.summary()

# Вызначэнне EarlyStopping для прадухілення перавучэння
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Падзяляем навучальную выбарку на навучальную і валідацыйную (храналагічна)
val_split_idx = int(len(X_train) * 0.85)
X_tr, y_tr = X_train[:val_split_idx], y_train[:val_split_idx]
X_val, y_val = X_train[val_split_idx:], y_train[val_split_idx:]

# Навучанне мадэлі
history = model.fit(
    X_tr, y_tr,
    epochs=50,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=1
)

Model: "sequential"
┌─────────────────────────────────┬────────────────────────┬───────────────┐
│ Layer (type)                    │ Output Shape           │       Param # │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 64)             │        12,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘
 Total params: 12,929 (50.50 KB)
 Trainable params: 12,929 (50.50 KB)
 Non-trainable params: 0 (0.00 B)
Epoch 1/50

 1/48 ━━━━━━━━━━━━━━━━━━━━ 8:46 11s/step - loss: 0.1448
 3/48 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - loss: 0.1343 
 5/48 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - loss: 0.1245
 7/4

### Візуалізацыя працэсу навучання НС
Пабудуем графік змены памылкі на навучальнай і валідацыйнай выбарках па epoch.

In [10]:
# Пабудова графіка памылак навучання
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Памылка навучання (Train Loss)', color='royalblue', lw=2)
plt.plot(history.history['val_loss'], label='Памылка валідацыі (Val Loss)', color='darkorange', lw=2)
plt.title('Працэс навучання GRU мадэлі', fontsize=14)
plt.xlabel('Эпоха', fontsize=12)
plt.ylabel('Сярэдняквадратычная памылка (MSE)', fontsize=12)
plt.legend(fontsize=11)
plt.savefig('training_history.png', dpi=300)
plt.show()

### Прагназаванне на тэставай выбарцы з GRU
Зробім прагнозы і выканаем адваротнае пераўтварэнне маштабу (inverse transform), каб вярнуць іх да арыгінальных адзінак вымярэння CO2.

In [11]:
# Генерацыя прагнозаў на тэставым наборы
gru_predictions_scaled = model.predict(X_test)

# Адваротнае пераўтварэнне маштабу для прагнозаў і сапраўдных значэнняў
gru_predictions = scaler.inverse_transform(gru_predictions_scaled).flatten()
y_test_original = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

print("Першыя 5 прагнозаў GRU:", gru_predictions[:5])
print("Першыя 5 сапраўдных значэнняў:", y_test_original[:5])


 1/15 ━━━━━━━━━━━━━━━━━━━━ 32s 2s/step
 5/15 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
 8/15 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
11/15 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
14/15 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 186ms/step
15/15 ━━━━━━━━━━━━━━━━━━━━ 5s 197ms/step
Першыя 5 прагнозаў GRU: [358.9583 359.2575 359.2186 359.4531 359.9812]
Першыя 5 сапраўдных значэнняў: [359.1 358.8 359.4 360.  359.6]


## 4. Рэалізацыя мадэлі Facebook Prophet
Ініцыялізуем і навучым мадэль Prophet на навучальным датафрэйме. Мы дадамо штогадовую і штотыднёвую сезоннасць (хоць для тыднёвых даных тыднёвая сезоннасць менш значная, Prophet аўтаматычна вызначыць яе структуру).

In [12]:
# Стварэнне і навучанне мадэлі Prophet
import logging
logging.getLogger('prophet').setLevel(logging.ERROR)

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False
)

# Навучанне мадэлі
prophet_model.fit(prophet_train)

# Стварэнне будучага датафрэйма для прагнозу (яго перыяд павінен цалкам пакрываць тэставую выбарку)
future_df = pd.DataFrame({'ds': test_df['date']})

# Стварэнне прагнозаў
prophet_forecast = prophet_model.predict(future_df)

# Выманне прадказаных значэнняў (yhat)
prophet_predictions = prophet_forecast['yhat'].values

print("Першыя 5 прагнозаў Prophet:", prophet_predictions[:5])

Першыя 5 прагнозаў Prophet: [360.09499433 360.3834243  360.62576096 360.8072136  360.93910498]


13:53:36 - cmdstanpy - INFO - Chain [1] start processing
13:53:43 - cmdstanpy - INFO - Chain [1] done processing


## 5. Ацэнка якасці і параўнанне мадэляў
Для аб'ектыўнага параўнання якасці абедзвюх мадэляў мы разлічым тры стандартныя метрыкі на тэставым перыядзе:
1. **MAE** (Mean Absolute Error) — сярэдняя абсалютная памылка.
2. **RMSE** (Root Mean Squared Error) — корань з сярэдняквадратычнай памылкі.
3. **MAPE** (Mean Absolute Percentage Error) — сярэдняя абсалютная працэнтная памылка.

In [13]:
# Функцыя для разліку метрык
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    return mae, rmse, mape

# Разлік метрык для абедзвюх мадэляў
mae_gru, rmse_gru, mape_gru = calculate_metrics(y_test_original, gru_predictions)
mae_prophet, rmse_prophet, mape_prophet = calculate_metrics(y_test_original, prophet_predictions)

# Складанне параўнальнай табліцы
metrics_comparison = pd.DataFrame({
    'Метрыка': ['MAE (ppm)', 'RMSE (ppm)', 'MAPE (%)'],
    'GRU (Нейронавая Сетка)': [f'{mae_gru:.4f}', f'{rmse_gru:.4f}', f'{mape_gru:.4f}%'],
    'Facebook Prophet': [f'{mae_prophet:.4f}', f'{rmse_prophet:.4f}', f'{mape_prophet:.4f}%']
})

# Адлюстраванне табліцы
print("Параўнанне метрык якасці прагнозаў на тэставым перыядзе:")
print(metrics_comparison.to_markdown(index=False))

Параўнанне метрык якасці прагнозаў на тэставым перыядзе:
| Метрыка    | GRU (Нейронавая Сетка)   | Facebook Prophet   |
|:-----------|:-------------------------|:-------------------|
| MAE (ppm)  | 0.3847                   | 0.7057             |
| RMSE (ppm) | 0.4925                   | 0.8664             |
| MAPE (%)   | 0.1056%                  | 0.1946%            |


## 6. Візуалізацыя вынікаў прагназавання
Пабудуем высакаякасны сумесны графік сапраўдных значэнняў рада, прагнозу GRU нейрасеткі і прагнозу Prophet на тэставым перыядзе. Гэта дазволіць нам наглядна параўнаць паводзіны абедзвюх мадэляў.

In [14]:
# Пабудова фінальнага параўнальнага графіка
plt.figure(figsize=(14, 7))

# Сапраўдныя значэнні
plt.plot(test_df['date'], y_test_original, label='Фактычныя значэнні (Actual)', color='black', lw=2, zorder=2)

# Прагноз GRU
plt.plot(test_df['date'], gru_predictions, label='Прагноз GRU (RNN)', color='royalblue', linestyle='--', lw=2, zorder=3)

# Прагноз Prophet
plt.plot(test_df['date'], prophet_predictions, label='Прагноз Prophet (Meta)', color='darkorange', linestyle='-.', lw=2, zorder=1)

# Налады графіка
plt.title('Параўнанне прагнозаў: GRU (Нейронавая Сетка) vs. Facebook Prophet', fontsize=16, fontweight='bold')
plt.xlabel('Дата', fontsize=12)
plt.ylabel('Канцэнтрацыя CO2 (ppm)', fontsize=12)
plt.legend(fontsize=12, loc='upper left')
plt.grid(True, which='both', linestyle=':', alpha=0.8)

# Сфакусуемся больш наглядна на перыядзе
plt.xlim(test_df['date'].min(), test_df['date'].max())

plt.tight_layout()
plt.savefig('predictions_comparison.png', dpi=300)
plt.show()

## 7. Аналітичнае заключэнне

На аснове праведзенага эксперыменту і атрыманых метрык можна зрабіць наступныя высновы аб параўнанні двух метадаў:

### 1. Інтэрпрэтацыя атрыманых вынікаў
- **GRU нейрасетка** паказала выдатную здольнасць адаптавацца да дробных ваганняў і лакальных зменаў у радзе. Дзякуючы паслядоўнаму аналізу слізгальных вокнаў даўжынёй 52 тыдні (1 год), яна здольная вельмі дакладна аптымізаваць прагнозы на адзін крок наперад.
- **Facebook Prophet** выдатна спраўляецца з вылучэннем глабальнага трэнду і стабільнай гадавой сезоннасці. Гэта адытыўная мадэль, якая раскладае рад на гладкі трэнд і перыядычныя складнікі. Яна менш адчувальная да шуму і рэзкіх выкідаў.

### 2. Сцэнарыі прымянення мадэляў

#### Калі лепш выкарыстоўваць Facebook Prophet:
1. **Невялікія наборы даных і хуткі старт:** Prophet працуе вельмі хутка і не патрабуе рэсурсаёмістага навучання.
2. **Наяўнасць складаных каляндарных эфектаў:** Мадэль ідэальна падыходзіць для задач, дзе важна ўлічваць святочныя дні, выходныя і спецыфічныя бізнес-падзеі (напрыклад, Чорная пятніца).
3. **Інтэрпрэтавальнасць:** Prophet раскладае прагноз на асобныя кампаненты (трэнд, сезоннасць, святы), што дазваляе лёгка патлумачыць прынятыя рашэнні бізнес-карыстальнікам.
4. **Наяўнасць пропускаў і выкідаў:** Мадэль вельмі ўстойлівая да адсутных значэнняў і не патрабуе складанай інтэрпаляцыі даных.

#### Калі лепш выкарыстоўваць Нейронавыя сеткі (LSTM/GRU):
1. **Вялікія аб'ёмы даных:** Нейрасеткі здольныя выяўляць вельмі складаныя нелінейныя ўзаемасувязі пры наяўнасці сотняў тысяч або мільёнаў назіранняў.
2. **Кароткатэрміновае прагназаванне з высокай дакладнасцю:** Пры рабоце са слізгальнымі вокнамі сеткі выдатна адсочваюць лакальныя ваганні і хутка рэагуюць на змены ў дынаміцы рада.
3. **Шматмерныя часавыя рады:** Калі для прагнозу мэтавай зменай выкарыстоўваюцца дадатковыя знешнія прыкметы (напрыклад, прагназаванне попыту на тавары з улікам надвор'я, акцый, коштаў канкурэнтаў адначасова), рэкурэнтныя сеткі паказваюць значна больш высокую гнуткасць у параўнанні з аднамерным Prophet.

### Вынік:
Абедзве мадэлі з'яўляеміся каштоўнымі інструментамі ў арсенале Data Scientist. Для простых бізнес-метрык са стабільнай сезоннасцю рэкамендуецца выкарыстоўваць **Prophet** як хуткае і інтэрпрэтавальнае рашэнне. Для буйных масіваў складаных нелінейных даных перавагу варта аддаць **LSTM/GRU** архітэктурам.